In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression

In [44]:
train = pd.read_csv('heart.csv')
X = train.drop('target', axis=1)
Y = train['target']

In [102]:
logreg = LogisticRegression(C=1e10, max_iter=1000)
logreg.fit(X, Y)
y_pred = logreg.predict_proba(X)[:, 1]
np.mean(-Y * np.log(y_pred) - (1 - Y) * np.log(1 - y_pred))

np.float64(0.34890516082100775)

In [75]:
def grad(w, X, Y):
    sigmoid = lambda z: 1 / (1 + np.exp(-z))
    predictions = sigmoid(-Y * np.dot(X, w))
    return np.mean(predictions[:, None] * (-Y[:, None] * X), axis=0)

def train(w, X, Y, mu=1e-4, T=10_000, stop=1e-2):
    losses = []
    for t in tqdm(np.arange(T)):
        loss = np.mean(np.log(1 + np.exp(-Y * np.dot(X, w))))
        losses.append(loss)

        gradient = grad(w, X, Y)
        i = np.argmax(np.abs(gradient))
        w = w - mu * gradient[i] * np.eye(X.shape[1])[i]

        tqdm.write(f"Step: {t+1}, Loss: {loss}, Grad: {abs(gradient[i])}", end="\r")
        if abs(gradient[i]) < stop:
            break
        
    return w, losses

In [76]:
w, losses = train(np.zeros(X.shape[1]), X.to_numpy(), Y.to_numpy(), T=50)
px.line(x=np.arange(len(losses))+1, y=losses, title="Coordinate Descent", labels={"x": "Step", "y": "Loss"})

  0%|          | 0/50 [00:00<?, ?it/s]